In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torchvision.models import ViT_B_16_Weights

from torch.utils.data import DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cpu


In [7]:
TRAIN_DIR = "../data/COVID_19_dataset/train"
VAL_DIR   = "../data/COVID_19_dataset/val"
TEST_DIR  = "../data/COVID_19_dataset/test"

In [8]:
def apply_clahe(img):

    img = np.array(img)

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    enhanced = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2RGB
    )

    return Image.fromarray(
        enhanced
    )

In [9]:
mean = [0.5159, 0.5159, 0.5159]
std  = [0.2280, 0.2280, 0.2280]
train_transform = transforms.Compose([

    transforms.Lambda(
        apply_clahe
    ),

    transforms.Resize(
        (224,224)
    ),

    transforms.RandomAffine(
        degrees=5,
        translate=(0.03,0.03)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])
test_transform = transforms.Compose([

    transforms.Lambda(
        apply_clahe
    ),

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [10]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

In [11]:
CLASS_NAMES = train_dataset.classes

print(CLASS_NAMES)

['COVID', 'Normal', 'Viral Pneumonia']


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
weights = ViT_B_16_Weights.IMAGENET1K_V1

model = vit_b_16(
    weights=weights
)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\Shantanu Ojha/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:30<00:00, 11.5MB/s] 


In [20]:
for param in model.parameters():

    param.requires_grad = False
for param in model.heads.parameters():

    param.requires_grad = True


In [21]:
model.heads.head = nn.Linear(
    768,
    3
)

In [22]:
model = model.to(device)

In [24]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    model.heads.parameters(),

    lr=1e-3
)

In [25]:
def train_one_epoch():

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(
            outputs,
            1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [26]:
def validate():

    model.eval()

    running_loss = 0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            _, preds = torch.max(
                outputs,
                1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    val_loss = running_loss / len(val_loader)

    val_acc = correct / total

    return val_loss, val_acc

In [ ]:
EPOCHS = 10
history = {

    "train_loss": [],
    "train_acc": [],

    "val_loss": [],
    "val_acc": []
}
for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Accuracy: {train_acc:.4f}"
    )

    print(
        f"Validation Loss: {val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: {val_acc:.4f}"
    )

In [ ]:
torch.save(

    model.state_dict(),

    "vit_b16.pth"
)

print("ViT-B/16 model saved.")